# Reproducing Li, Namburi, Olson et al. (2022) from dandiset 000689

**Paper.** Li H, Namburi P, Olson JM, et al. (2022). Neurotensin orchestrates valence assignment in
the amygdala. *Nature* 608, 586-592.
[doi:10.1038/s41586-022-04964-y](https://doi.org/10.1038/s41586-022-04964-y). The claims file was
written from the accepted author manuscript in PubMed Central (PMC9583860): main text, Methods
with the per-panel statistics, and the legends of Figures 1 to 5 and Extended Data Figures 1 to 10.

**Dandiset.** [DANDI:000689](https://dandiarchive.org/dandiset/000689/0.240530.1923), version
0.240530.1923. The paper cites no DANDI version (it predates the dandiset, created in October 2023).
There are two published versions, 0.240524.2012 and 0.240530.1923; the data check below shows that
both list the same 85 assets with the same asset identifiers, so they do not differ in content.
The most recent, 0.240530.1923, is pinned, as the protocol asks when the paper cites none.

This notebook tests the electrophysiology and NT sensor photometry claims, and shows why the
others are blocked.

**What the dandiset holds.** 31 sessions of BLA recordings during the three-cue discrimination
task (one per mouse, spike times of 1902 units, trial events, DeepLabCut pose in 29 of them), two
pose-only files for other sessions of two of those mice, and 20 photometry sessions of the
GRAB-NTS1.0 NT sensor in the BLA (5 mice, early and late acquisition of tone-sucrose and
tone-airpuff learning). No file says which mice are CRISPR and which are controls, nor which
projection (NAc or CeA) was tagged in each mouse, and the units are not the paper's sorted single
units (data check).

| Claim | Tested here | Outcome basis |
|---|---|---|
| fig3k-nt-sensor-sucrose-increase | yes | NT sensor files |
| fig3l-nt-sensor-airpuff-decrease | yes | NT sensor files |
| fig3-nt-sensor-early-dip | yes | NT sensor files |
| fig4-recording-count | yes | units table |
| fig4d-seven-clusters | run, blocked, methods_underspecified | units table |
| fig5dh-behavior-clusters | blocked, metadata_insufficient | pose cannot be aligned to the trials |
| fig4c, ed8d, ed8fg, fig4e, ed8i, fig4fg (2), fig5a, fig5b, fig5c, ed8e, fig5fi, ed8os | blocked, metadata_insufficient | no group or projection labels |
| fig1c, ed1c, ed2, ed3, ed5f, fig1g, fig1h, fig2c, fig2d, fig2g, fig2h, fig3bc, fig3e, fig3f, ed6, fig3h, ed9, ed9kl, ed10b, ed10 | blocked, data_missing | experiments not in the dandiset |

In [ ]:
import time, json, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

import nt_bla as nb

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
started = time.time()
A = nb.assets()
print(f"dandiset {nb.DANDISET} version {nb.VERSION}: {len(A)} assets")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
# The two published versions: same assets?
other = []
url = f"{nb.API}/dandisets/{nb.DANDISET}/versions/0.240524.2012/assets/?page_size=200"
while url:
    r = requests.get(url, timeout=60).json(); other += r["results"]; url = r.get("next")
a1 = {a["path"]: (a["asset_id"], a["size"]) for a in A}
a2 = {a["path"]: (a["asset_id"], a["size"]) for a in other}
print("assets in 0.240530.1923:", len(a1), " in 0.240524.2012:", len(a2),
      " same paths:", set(a1) == set(a2), " same asset ids and sizes:", a1 == a2)
kind = lambda p: ("photometry nwb" if "ophys" in p else "ecephys nwb" if "ecephys" in p and p.endswith(".nwb")
                  else "pose-only nwb" if p.endswith(".nwb") else "video")
inv = pd.DataFrame({"path": list(a1), "GB": [a1[p][1] / 1e9 for p in a1]})
inv["kind"] = inv.path.map(kind)
inv["subject"] = inv.path.str.split("/").str[0]
print(inv.groupby("kind").agg(files=("path", "size"), subjects=("subject", "nunique"), GB=("GB", "sum")).round(1).to_string())

The two published versions hold the same assets, so the choice between them cannot change a result.
The archive's summary counts 36 subjects: 31 electrophysiology mice and 5 photometry mice.

In [ ]:
# Electrophysiology: spike times and trial tables of the 31 sessions (raw voltage is not read).
EP = sorted(a["path"] for a in A if a["path"].endswith(".nwb") and "ecephys" in a["path"])
t = time.time()
with ThreadPoolExecutor(4) as ex:
    E = list(ex.map(nb.load_ephys, EP))
print(f"loaded {len(E)} sessions in {time.time() - t:.0f} s")
rows = []
for e in E:
    tr = e["trials"]
    r = {"subject": e["subject"], "session": e["session_id"][-22:], "genotype": e["genotype"],
         "units": len(e["spikes"]), "channels": e["n_channels"], "dur_s": round(e["duration"])}
    for k, v in nb.CS_TYPES.items():
        r[k] = len(tr[v][0]) if v in tr else 0
    r["sucrose_deliv"] = len(tr.get("reward_delivery", ([],))[0])
    r["shock_relay"] = len(tr.get("shock_relay", ([],))[0])
    r["light"] = len(tr.get("phototagging", ([],))[0])
    cs = np.concatenate([tr[v][0] for v in nb.CS_TYPES.values() if v in tr])
    r["task_end_s"] = round(cs.max()); r["light_start_s"] = round(tr["phototagging"][0].min())
    rows.append(r)
ses = pd.DataFrame(rows)
print(ses.to_string(index=False))
print("\nmice:", ses.subject.nunique(), " sessions:", len(ses), " units:", ses.units.sum(),
      " genotype values:", sorted(ses.genotype.unique()))
print("sessions without neutral CS trials:", list(ses.subject[ses.neutral == 0]))
print("CS counts (sucrose, neutral, shock) in the other sessions:",
      ses[ses.neutral > 0][["sucrose", "neutral", "shock"]].apply(tuple, axis=1).value_counts().to_dict())

**Checked against the paper.** The session description says sucrose, neutral and shock CSs at 2:1:1,
and 25 sessions hold 60, 30 and 30 CSs (some with one or two fewer), as the Methods give (120
trials). Six sessions (H09 to H14, recorded on 2019-08-30, "Disc4") hold 75 sucrose and 45 shock CSs
and no neutral CS: the description does not fit them. Every file gives the genotype as "Wild type";
none says whether the mouse is in the CRISPR or the control group, nor whether Cav2-Cre was put in
the NAc or the CeA, which the group comparisons and the phototagging need. There are 31 mice where
the paper reports 30. H28 and H32 also have a pose-only file for a later discrimination day (Disc4,
Disc6) than their recording (Disc3, Disc5), so their recorded session may not be the last day,
which is the only day the paper analyses. Each session ends with about 850 light pulses after the
task (the phototagging session); no separate "fake" session is labelled.

In [ ]:
# Are the units sorted single units? Refractory violations and spikes shared between units.
def coinc(a, b, w=1e-4):
    if len(a) == 0 or len(b) < 2: return 0.0
    j = np.clip(np.searchsorted(b, a), 1, len(b) - 1)
    return float(np.mean(np.minimum(np.abs(b[j] - a), np.abs(b[j - 1] - a)) < w))
q = []
for e in E:
    sp = e["spikes"]
    isi = [np.mean(np.diff(s) < 0.001) if len(s) > 1 else np.nan for s in sp]
    allsp = np.concatenate(sp)
    shared = 1 - len(np.unique(np.round(allsp * 30000).astype(np.int64))) / len(allsp)
    nn = min(len(sp), 16)
    pair = [coinc(sp[i], sp[j]) for i in range(nn) for j in range(nn) if i != j]
    q.append({"subject": e["subject"], "units": len(sp), "median_frac_isi_lt_1ms": np.nanmedian(isi),
              "units_with_isi_viol_gt_5pct": int(np.sum(np.array(isi) > 0.05)),
              "frac_spike_samples_shared": shared, "median_pair_coincidence_0.1ms": np.median(pair)})
q = pd.DataFrame(q)
print(q.round(3).to_string(index=False))
print("\nover all units: median fraction of ISIs below 1 ms = %.3f; units with more than 5%% = %d of %d"
      % (q.median_frac_isi_lt_1ms.median(), q.units_with_isi_viol_gt_5pct.sum(), q.units.sum()))
print("units per session: %s" % q.units.value_counts().sort_index().to_dict())

**Checked against the paper.** The Methods say single units were sorted in Offline Sorter, and the
paper analyses 685 neurons from 30 mice (about 23 per mouse). The files hold 1902 units, 64 in 21
of 31 sessions (two per channel of the 32-channel optrode). In 1061 units more than 5% of the
inter-spike intervals are below 1 ms (a median of 17 to 35% per session in H01 to H14), which a
well-isolated single neuron does not show, and in each session 6 to 65% of spike samples recur in
another unit. These are not the paper's sorted single units:
they look like unsorted or multi-unit threshold crossings. The claims that use units are tested on
them as they are, and this is recorded as a finding.

In [ ]:
# Clock check for the spikes and the trial table: CS-evoked change in population rate at the
# true CS onsets against the same onsets shifted 10 s earlier (inside the inter-trial interval).
def pop_resp(e, shift):
    on = np.concatenate([e["trials"][v][0] for v in nb.CS_TYPES.values() if v in e["trials"]]) + shift
    out = []
    for sp in e["spikes"]:
        p = nb.psth(sp, on, 1.0, 1.0, 0.1)
        b = p[:10].mean()
        out.append((p[10:13].mean() - b) / (p[:10].std(ddof=1) + 1e-9))
    return np.median(np.abs(out))
ck = pd.DataFrame([{"subject": e["subject"], "true_onsets": pop_resp(e, 0.0), "shifted_-10s": pop_resp(e, -10.0)} for e in E])
print("median |z| of the 0 to 0.3 s response across units, per session:")
print(ck.round(2).to_string(index=False))
print("sessions where the true onsets give the larger response: %d of %d" % ((ck.true_onsets > ck["shifted_-10s"]).sum(), len(ck)))

The units respond more at the true CS onsets than at onsets shifted into the inter-trial interval in
the sessions counted above, which is consistent with the spikes and the trial table sharing a clock
(a check that supports, without proving, the alignment). It fails in H17, whose spike times end
at 249 s of a 12118 s recording (next cell): its units cannot be aligned with its trials. In H22,
H27, H34 and H39 the two values are close, so the alignment is not shown there.

In [ ]:
# Span of the spike times against the length of the recording, per session.
span = pd.DataFrame([{"subject": e["subject"], "first_spike_s": round(min(s[0] for s in e["spikes"] if len(s)), 1),
                      "last_spike_s": round(max(s[-1] for s in e["spikes"] if len(s)), 1), "recording_s": round(e["duration"], 1),
                      "task_start_s": round(min(e["trials"][v][0].min() for v in nb.CS_TYPES.values() if v in e["trials"]), 1),
                      "task_end_s": round(max(e["trials"][v][0].max() for v in nb.CS_TYPES.values() if v in e["trials"]), 1),
                      "light_before_task": int(np.sum(e["trials"]["phototagging"][0] < min(e["trials"][v][0].min() for v in nb.CS_TYPES.values() if v in e["trials"]))),
                      "light_after_task": int(np.sum(e["trials"]["phototagging"][0] > max(e["trials"][v][0].max() for v in nb.CS_TYPES.values() if v in e["trials"])))} for e in E])
print(span.to_string(index=False))
print("sessions whose spikes stop before 90% of the recording:", list(span.subject[span.last_spike_s < 0.9 * span.recording_s]))
print("sessions with light events before the task:", list(span.subject[span.light_before_task > 0]))
# Light events form two blocks per session: split at the largest gap between events.
for e in E[:3] + E[-3:]:
    x = e["trials"]["phototagging"][0]; g = np.argmax(np.diff(x))
    print(f'{e["subject"]}: blocks of {g + 1} and {len(x) - g - 1} light events, gap {np.diff(x)[g]:.0f} s')

H17's spike times all fall within the first 249 s of a recording of 12118 s, while its trial events
cover the whole session: its spike times are not on the trial clock (a file error). Every session
holds about 854 light events in two blocks of about 427, which fits the Methods' two light sessions
(a "fake" one with the patch cord disconnected, then the real one), but no label says which block
is the fake session. In 12 sessions (H31 to H42) one block comes before the task and one after it;
the Methods put both after the task.

In [ ]:
# Does the pose share the trial clock? The pose series give only a start time (0 s) and a nominal
# 15 frames/s, no timestamps. At each port entry (beam break, trial table) the head should be at
# the port, so the lag (pose time minus trial time) that minimises the median head-to-port
# distance at port entries is estimated separately in each third of the session.
POSE_P = [p for p in EP]
def pose_lag(p):
    h = nb.open_h5(p)
    if "processing/behavior/PoseEstimation" not in h:
        return {"subject": p.split("/")[0][4:], "pose": False}
    pose = nb.load_pose(h); r = pose["rate"]
    dist = nb.pose_measures(pose)["distance_to_port"]
    tt = h["intervals/trials/trial_type"][:].astype(str); st = h["intervals/trials/start_time"][:]
    pe = st[tt == "port_entry"]
    lags = np.arange(-200, 200.01, 0.2)
    out = {"subject": p.split("/")[0][4:], "pose": True, "pose_s": round(len(dist) / r), "port_entries": len(pe)}
    for k, seg in enumerate(np.array_split(pe, 3)):
        med = []
        for L in lags:
            f = np.round((seg + L - pose["t0"]) * r).astype(int); f = f[(f >= 0) & (f < len(dist))]
            med.append(np.nanmedian(dist[f]) if len(f) > 5 else np.nan)
        med = np.array(med)
        out[f"lag_third{k + 1}_s"] = round(float(lags[np.nanargmin(med)]), 1)
        out[f"dist_ratio_third{k + 1}"] = round(float(np.nanmin(med) / med[np.argmin(np.abs(lags))]), 2)
    return out
with ThreadPoolExecutor(4) as ex:
    pl = pd.DataFrame(list(ex.map(pose_lag, POSE_P)))
print(pl.to_string(index=False))
ok = pl[pl.pose]
spread = ok[["lag_third1_s", "lag_third2_s", "lag_third3_s"]].max(axis=1) - ok[["lag_third1_s", "lag_third2_s", "lag_third3_s"]].min(axis=1)
print("\nsessions with pose: %d; median |lag| %.1f s; sessions whose best lag changes by more than 5 s within the session: %d"
      % (len(ok), np.median(np.abs(ok[["lag_third1_s", "lag_third2_s", "lag_third3_s"]].values)), int((spread > 5).sum())))

The pose cannot be placed on the trial clock from the files. 29 recorded sessions have pose (H28
and H32 have pose only for another day, in files with no trial table). If the pose were on the
trial clock, the head would be closest to the port at lag 0; instead the best lag is tens of
seconds away in most sessions and drifts within the session (the table), as a video whose real
frame rate differs from the nominal 15 frames per second would. "dist_ratio" is the median
head-to-port distance at the best lag over that at lag 0. The behavior clustering claim, which
cuts the 20 s after each CS from the pose, is blocked as `metadata_insufficient`.

In [ ]:
# Photometry: the 20 NT sensor sessions.
PP = sorted(a["path"] for a in A if "ophys" in a["path"])
t = time.time()
with ThreadPoolExecutor(4) as ex:
    S = list(ex.map(nb.load_photometry, PP))
print(f"loaded {len(S)} sessions in {time.time() - t:.0f} s")
ph = []
for s in S:
    sid = s["session_id"]
    lick_rel = np.concatenate([s["licks"][(s["licks"] > c - 2) & (s["licks"] < c + 6)] - c for c in s["cs_on"]])
    ph.append({"subject": s["subject"], "task": "sucrose" if "Reward" in sid else "airpuff",
               "phase": "early" if "early" in sid else "late", "fluorophore": s["fluorophore"][:14],
               "fiber": s["fiber_location"], "nm": s["wavelengths"], "frames": s["n_frames"],
               "dt_ms": tuple(round(x * 1000, 1) for x in s["frame_dt"]),
               "phase_means": tuple(round(x, 3) for x in s["phase_means"]),
               "ph415": s["phase415"], "ph470": s["phase470"],
               "agree": round(min(s["phase_agree415"], s["phase_agree470"]), 3),
               "cues": len(s["cs_on"]), "cue_s": round(float(np.median(s["cs_dur"])), 2),
               "min_gap_s": round(float(np.diff(s["cs_on"]).min()), 1),
               "lick_frames_-2_0": int(((lick_rel >= -2) & (lick_rel < 0)).sum()),
               "lick_frames_2_4": int(((lick_rel >= 2) & (lick_rel < 4)).sum())})
ph = pd.DataFrame(ph)
print(ph.to_string(index=False))

**Checked against the paper.** The files hold the GRAB-NTS1.0 sensor ("neurotensin-fluorescent
sensor") in the BLA of five wild-type mice, with 415 and 470 nm excitation, which is the experiment
of Figure 3j-l (N = 5; the Methods exclude one of six mice for fiber placement, and the subjects are
Mouse1, 2, 4, 5 and 6). Frames come every 16.7 ms (60 Hz), not at the Methods' 40 Hz: one series
interleaves three frame types. The LED labels mark every 415 nm frame and every 470 nm frame (a third
of the frames each); these usually sit on a fixed phase of three ("agree" is the fraction of
labelled frames on the most common phase), but in Mouse6's late sucrose session the phase shifts
after a 2.3 ms frame interval, so channels are taken from the labels, not from the frame order.
The third frame type is a near-constant value around 0.13 to 0.20 that no metadata describes. Each session holds about
60 cues (30 in Mouse1's early airpuff session) lasting 1.9 to 2 s, at least 29 s apart. The Methods
give a 4 s CS with the US 2 s after onset and about half of the CSs unpaired; the files mark one
kind of cue, 2 s long, and no US. In sucrose sessions licking rises after the cue ends (frames with
a lick 2 to 4 s after onset against the 2 s before), which places the sucrose at the end of the 2 s
cue, as the Methods' US time of 2 s gives; no file marks whether a cue was paired or unpaired, nor
the airpuff. All cues are therefore analysed as CS-US trials, an assumption recorded with the
claims.

In [ ]:
# Per-trial sucrose check: in the late sucrose sessions, how many cues are followed by licking
# within 2 to 5 s of onset (a paired trial should be, an unpaired one need not)?
for s in S:
    if "Reward" in s["session_id"]:
        n = [np.sum((s["licks"] >= c + 2) & (s["licks"] < c + 5)) > 0 for c in s["cs_on"]]
        print(f'{s["session_id"]:34s} cues followed by a lick within 2-5 s: {sum(n)} of {len(n)}')

In the late sucrose sessions most cues are followed by licking, so the files' cues are mostly, and
perhaps all, paired with sucrose; the early sessions, before learning, cannot tell.

## Photometry claims (Figure 3j-l)

Processing as the claims' criterion gives it (Methods, Fiber photometry, Data analysis): 415 and
470 nm channels median-filtered over 200 ms and averaged into 200 ms (5 Hz) bins from -5 to 10 s
around each CS onset; the trial-averaged 415 nm trace regressed on the trial-averaged 470 nm trace
over -5 to 0 s, the residuals z-scored against -5 to 0 s. Windows: CS 0 to 2 s, US 2 to 7 s (Figure 3
legend). Early and late are the files' early and late acquisition sessions. Readings run, as the
criterion lists them: (a) first 16 trials or (b) all trials; (i) area under the z-scored curve or
(ii) mean z. One more detail the Methods leave open is run both ways: the z-score uses each trial's
own baseline ("trial") or the session's pooled baseline ("session").

In [ ]:
TRIALS = {}
for s in S:
    for zm in ("trial", "session"):
        tb, Z, keep = nb.photometry_trials(s, zmode=zm)
        TRIALS[(s["subject"], "sucrose" if "Reward" in s["session_id"] else "airpuff",
                "early" if "early" in s["session_id"] else "late", zm)] = (tb, Z)
print("trials kept per session (all cues have a full -5 to 10 s window):",
      {f"{k[0]} {k[1]} {k[2]}": v[1].shape[0] for k, v in TRIALS.items() if k[3] == "trial"})
mice = sorted({k[0] for k in TRIALS})

def measure(task, phase, zm, sub, meas, lo, hi):
    out = []
    for m in mice:
        tb, Z = TRIALS[(m, task, phase, zm)]
        Z = Z[:16] if sub == "first16" else Z
        out.append(nb.window_measure(tb, Z, lo, hi, meas).mean())
    return np.array(out)

def compare(task):
    rows = []
    for zm in ("trial", "session"):
        for sub in ("first16", "all"):
            for meas in ("auc", "mean"):
                r = {"zscore": zm, "trials": sub, "measure": meas}
                for win, (lo, hi) in (("US", (2, 7)), ("CS", (0, 2))):
                    e, l = measure(task, "early", zm, sub, meas, lo, hi), measure(task, "late", zm, sub, meas, lo, hi)
                    tt = stats.ttest_rel(l, e)
                    d = l - e
                    ci = stats.t.interval(0.95, len(d) - 1, d.mean(), stats.sem(d))
                    r.update({f"{win}_late-early": d.mean(), f"{win}_ci_low": ci[0], f"{win}_ci_high": ci[1],
                              f"{win}_t": tt.statistic, f"{win}_p": tt.pvalue, f"{win}_mice_in_direction": None})
                    r[f"{win}_mice_up"] = int((d > 0).sum())
                rows.append(r)
    return pd.DataFrame(rows).drop(columns=["US_mice_in_direction", "CS_mice_in_direction"])

### fig3k-nt-sensor-sucrose-increase

> The BLA NT sensor response to sucrose delivery increases after reward learning.
> (Paper: US window late against early, paired t-test t(4) = 8.709, P = 0.001, effect 4.14, CI 2.744
> to 5.538; CS window t(4) = 1.788, P = 0.1483.) Reproduced when the US comparison has p < 0.05 with
> late higher under every reading.

In [ ]:
res_suc = compare("sucrose")
print(res_suc.round(4).to_string(index=False))
print("\nPer mouse, US window AUC (trial z-score, first 16 trials):")
print(pd.DataFrame({"mouse": mice, "early": measure("sucrose", "early", "trial", "first16", "auc", 2, 7),
                    "late": measure("sucrose", "late", "trial", "first16", "auc", 2, 7)}).round(2).to_string(index=False))
print("\nReadings with US p < 0.05 and late higher: %d of %d" % (((res_suc["US_p"] < 0.05) & (res_suc["US_late-early"] > 0)).sum(), len(res_suc)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, task in zip(axes, ("sucrose", "airpuff")):
    for phase, col in (("early", "tab:blue"), ("late", "tab:red")):
        tr = np.array([TRIALS[(m, task, phase, "trial")][1][:16].mean(0) for m in mice])
        tb = TRIALS[(mice[0], task, phase, "trial")][0] + 0.1
        ax.plot(tb, tr.mean(0), color=col, label=phase)
        ax.fill_between(tb, tr.mean(0) - stats.sem(tr), tr.mean(0) + stats.sem(tr), color=col, alpha=0.2)
        print(f"{task} {phase}: plotted mean z from {tr.mean(0).min():.2f} to {tr.mean(0).max():.2f}")
    ax.axvspan(2, 7, color="0.9"); ax.axvline(0, color="k", lw=0.5)
    ax.set_title(f"NT sensor, {task} (first 16 trials, mean of 5 mice)"); ax.set_xlabel("s from CS onset")
axes[0].set_ylabel("z"); axes[0].legend(); plt.tight_layout(); plt.show()

**Result: not reproduced.** Under the per-trial z-score the US response to sucrose is larger in the
late session under all four readings of the criterion (AUC +3.01 with the first 16 trials and +2.19
with all trials), in the direction the paper reports and near its effect of 4.14, but no reading
reaches p < 0.05 (smallest p = 0.081, with 4 of 5 mice higher). Under the session-pooled z-score the
difference vanishes (AUC -0.76 to +0.08, p >= 0.64). The CS window does not change under any
reading (p >= 0.49), as in the paper.

### fig3l-nt-sensor-airpuff-decrease

> The BLA NT sensor response to airpuff decreases after punishment learning.
> (Paper: US window t(4) = 3.996, P = 0.0162, effect -3.52, CI -6.083 to -0.9583; CS t(4) = 2.137,
> P = 0.0994.) Reproduced when the US comparison has p < 0.05 with late lower under every reading.

In [ ]:
res_air = compare("airpuff")
print(res_air.round(4).to_string(index=False))
print("\nPer mouse, US window AUC (trial z-score, first 16 trials):")
print(pd.DataFrame({"mouse": mice, "early": measure("airpuff", "early", "trial", "first16", "auc", 2, 7),
                    "late": measure("airpuff", "late", "trial", "first16", "auc", 2, 7)}).round(2).to_string(index=False))
print("\nReadings with US p < 0.05 and late lower: %d of %d" % (((res_air["US_p"] < 0.05) & (res_air["US_late-early"] < 0)).sum(), len(res_air)))

**Result: not reproduced.** The US response to airpuff is lower in the late session under every
reading (AUC -5.47 with the first 16 trials and -4.66 with all trials under the per-trial z-score,
near the paper's -3.52), but no reading reaches p < 0.05 (smallest p = 0.074). The CS window does
not change significantly under any reading (p >= 0.08).

### fig3-nt-sensor-early-dip

> Early in acquisition, the BLA NT sensor signal transiently decreases after both sucrose and
> airpuff. Reproduced when the mean z-scored signal over mice in the US window (2 to 7 s from CS
> onset) is below zero in early trials for both sucrose and airpuff.

In [ ]:
rows = []
for zm in ("trial", "session"):
    for sub in ("first16", "all"):
        for meas in ("mean", "auc"):
            r = {"zscore": zm, "trials": sub, "measure": meas}
            for task in ("sucrose", "airpuff"):
                v = measure(task, "early", zm, sub, meas, 2, 7)
                r[f"{task}_mean"] = v.mean(); r[f"{task}_mice_below_0"] = int((v < 0).sum())
                r[f"{task}_p_vs_0"] = stats.ttest_1samp(v, 0).pvalue
            rows.append(r)
dip = pd.DataFrame(rows)
print(dip.round(3).to_string(index=False))
print("\nReadings with both early means below zero: %d of %d" % (((dip.sucrose_mean < 0) & (dip.airpuff_mean < 0)).sum(), len(dip)))
print("\nEarly sessions, per mouse, mean z 2-7 s (trial z-score, first 16 trials):")
print(pd.DataFrame({"mouse": mice, "sucrose": measure("sucrose", "early", "trial", "first16", "mean", 2, 7),
                    "airpuff": measure("airpuff", "early", "trial", "first16", "mean", 2, 7)}).round(2).to_string(index=False))

**Result: not reproduced.** In the early sessions the mean over mice of the US-window signal is
below zero for sucrose under the per-trial z-score (-0.62 with the first 16 trials, -0.40 with all
trials) but above zero for airpuff (+0.11 and +0.19), so the criterion, which needs both below zero,
is not met under the first reading it lists or under any per-trial reading. Under the
session-pooled z-score the signs change with the trial subset (see the table).

## Electrophysiology claims (Figure 4)

### fig4-recording-count

> 685 BLA neurons were recorded from 30 mice during the three-cue discrimination task. Units from
> the last discrimination-task session of each mouse, with baseline firing of at least 0.3 Hz
> (Methods). Whether 685 is counted before or after that exclusion is not said; both counts are
> shown and either may match (expected 685 +/- 68.5 neurons, 30 +/- 3 mice).

Baseline rate: the mean rate in the 1 s before each CS onset, over all CSs of the session (the PSTH
baseline of the Methods); the whole-session rate is shown as a variant.

In [ ]:
base_rate, sess_rate, unit_rows = {}, {}, []
for e in E:
    on = np.sort(np.concatenate([e["trials"][v][0] for v in nb.CS_TYPES.values() if v in e["trials"]]))
    for u, sp in enumerate(e["spikes"]):
        b = np.mean(np.searchsorted(sp, on) - np.searchsorted(sp, on - 1.0))
        unit_rows.append({"subject": e["subject"], "unit": u, "baseline_hz": b, "session_hz": len(sp) / e["duration"]})
U = pd.DataFrame(unit_rows)
n_all, n_base, n_sess = len(U), int((U.baseline_hz >= 0.3).sum()), int((U.session_hz >= 0.3).sum())
print(f"units before exclusion: {n_all}; with baseline >= 0.3 Hz: {n_base}; with session rate >= 0.3 Hz: {n_sess}")
print(f"mice: {U.subject.nunique()}")
print("per mouse (all units / kept by baseline):")
print(U.groupby("subject").agg(units=("unit", "size"), kept=("baseline_hz", lambda x: int((x >= 0.3).sum()))).T.to_string())
print(f"\nexpected 685 +/- 68.5 neurons: before exclusion {abs(n_all - 685) <= 68.5}, after {abs(n_base - 685) <= 68.5}")
print(f"expected 30 +/- 3 mice: {abs(U.subject.nunique() - 30) <= 3}")
print("units per mouse: median %.0f (paper 685 / 30 = 22.8)" % U.groupby("subject").size().median())

**Result: not reproduced.** The dandiset holds 1902 units from 31 mice, 1739 of them with a
baseline rate of at least 0.3 Hz (1884 by whole-session rate), against the paper's 685 neurons;
neither count is within 68.5 of 685. The number of mice (31) is within the tolerance of 30. The units
are not sorted single units (data check), which explains a count near 64 per mouse instead of 23.

### fig4d-seven-clusters

> Unsupervised hierarchical clustering of BLA responses to the sucrose, neutral and shock CSs gives
> seven functional clusters. Per neuron, 20 ms PSTHs from -1 to 1 s around CS onset, z-scored with
> the 1 s baseline, smoothed with a Gaussian-weighted moving average (smoothing factor 0.85), the
> three CS traces concatenated; Ward linkage with correlation distance cut at 35% of the maximum
> linkage distance; readings (a) Ward on the correlation distance matrix, (b) Ward with Euclidean
> distance on centred, unit-norm rows. Reproduced when one reading gives seven clusters.

Units with a baseline rate below 0.3 Hz are excluded (Methods). The six sessions without neutral
CSs cannot give the concatenated trace and are left out; so are units whose baseline SD is zero.
The "smoothing factor of 0.85" sets the window through a MATLAB heuristic the paper does not give,
so the smoothing window is run over a range (in 20 ms bins, smoothing each CS trace separately).

In [ ]:
feats, owner = [], []
dropped = {"low_rate": 0, "no_neutral_session": 0, "zero_sd": 0}
for e in E:
    on = {k: e["trials"][v][0] for k, v in nb.CS_TYPES.items() if v in e["trials"]}
    allon = np.sort(np.concatenate(list(on.values())))
    for u, sp in enumerate(e["spikes"]):
        b = np.mean(np.searchsorted(sp, allon) - np.searchsorted(sp, allon - 1.0))
        if b < 0.3: dropped["low_rate"] += 1; continue
        if "neutral" not in on: dropped["no_neutral_session"] += 1; continue
        z = np.array([nb.zscore_psth(nb.psth(sp, on[k])) for k in ("sucrose", "neutral", "shock")])
        if np.isnan(z).any(): dropped["zero_sd"] += 1; continue
        feats.append(z); owner.append(e["subject"])
F = np.array(feats); owner = np.array(owner)
print("units clustered:", len(F), "from", len(set(owner)), "mice; dropped:", dropped)

def cluster(F, window):
    X = nb.gauss_smooth(F, window).reshape(len(F), -1)
    Za = linkage(pdist(X, "correlation"), "ward")
    ca = fcluster(Za, 0.35 * Za[:, 2].max(), "distance")
    Xc = X - X.mean(1, keepdims=True); Xc /= np.linalg.norm(Xc, axis=1, keepdims=True)
    Zb = linkage(Xc, "ward")
    cb = fcluster(Zb, 0.35 * Zb[:, 2].max(), "distance")
    return ca, cb

rows = []
CL = {}
for w in (1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 51, 75, 101):
    ca, cb = cluster(F, w)
    CL[w] = (ca, cb)
    rows.append({"window_bins": w, "window_ms": w * 20, "clusters_a_corr_ward": ca.max(), "clusters_b_unitnorm_ward": cb.max(),
                 "sizes_a": sorted(np.bincount(ca)[1:].tolist(), reverse=True)})
cl = pd.DataFrame(rows)
print(cl.to_string(index=False))
print("\nwindows giving 7 clusters: reading (a)", list(cl.window_bins[cl.clusters_a_corr_ward == 7]),
      " reading (b)", list(cl.window_bins[cl.clusters_b_unitnorm_ward == 7]))

In [ ]:
# Cut-off sensitivity for one window (11 bins = 220 ms): clusters as a function of the cut fraction.
Xw = nb.gauss_smooth(F, 11).reshape(len(F), -1)
Za = linkage(pdist(Xw, "correlation"), "ward")
print("reading (a), window 11 bins: clusters by cut fraction of the maximum linkage distance")
print({f: int(fcluster(Za, f * Za[:, 2].max(), "distance").max()) for f in (0.25, 0.3, 0.35, 0.4, 0.45, 0.5)})
# Per mouse composition of the clusters, reading (a), window 11.
ca = CL[11][0]
print("\nunits per mouse in each cluster (reading (a), window 11):")
print(pd.crosstab(owner, ca).to_string())
fig, ax = plt.subplots(figsize=(7, 4))
order = np.argsort(ca, kind="stable")
im = ax.imshow(Xw[order], aspect="auto", cmap="RdBu_r", vmin=-5, vmax=5, interpolation="nearest")
ax.set_xticks([0, 50, 100, 150, 200, 250]); ax.set_xticklabels(["S -1", "S 0", "N -1", "N 0", "Sh -1", "Sh 0"])
ax.set_ylabel("unit (sorted by cluster)"); plt.colorbar(im, label="z (clipped at +/-5)"); plt.show()
print("plotted z from %.1f to %.1f; %.1f%% of values outside +/-5" % (Xw.min(), Xw.max(), 100 * np.mean(np.abs(Xw) > 5)))

**Result: blocked (methods_underspecified).** With the paper's rule (Ward linkage, correlation
distance, cut at 35% of the maximum linkage distance) the 1399 units of 24 mice give 8 to 12
clusters under reading (a) and 11 to 14 under reading (b) for smoothing windows of 20 to 420 ms,
and 6 to 8 under (a) for windows of 620 ms to 2 s; seven clusters come only under reading (a) with
an 820 ms window. The count depends on the smoothing window, which the paper sets by MATLAB's
"smoothing factor of 0.85" without the window it produced, so no reading can be preferred; the
count also moves with the cut fraction (13 at 30%, 10 at 35%, 7 at 40% for a 220 ms window). The
units are not the paper's sorted neurons.

## Blocked claims

No file says which mice are CRISPR and which control, nor which projection (NAc or CeA) carries the
Cav2-Cre in each mouse. Every claim that compares the groups, or that needs BLA-NAc or BLA-CeA
labels, is blocked as `metadata_insufficient`. For the phototagging claim (fig4c), the light pulses
are in the files, so the photo-responsive rule of the Methods can still be run to show how many
units it would tag over both light blocks; without knowing which block is the fake session, and the
projection of each mouse, it cannot be compared with the 39 BLA-NAc and 32 BLA-CeA neurons.

In [ ]:
tag_rows = []
for e in E:
    on, off = e["trials"]["phototagging"]
    dur = off - on
    pulses = on[dur <= 0.011]
    one_hz = pulses[(np.r_[np.inf, np.diff(pulses)] > 0.9) & (np.r_[np.diff(pulses), np.inf] > 0.9)]
    n_all = n_1 = 0
    for sp in e["spikes"]:
        n_all += nb.photoresponsive(sp, pulses)[0]
        n_1 += nb.photoresponsive(sp, one_hz)[0]
    tag_rows.append({"subject": e["subject"], "pulses_5_10ms": len(pulses), "isolated_pulses": len(one_hz),
                     "tagged_all_pulses": n_all, "tagged_isolated_pulses": n_1, "units": len(e["spikes"])})
tg = pd.DataFrame(tag_rows)
print(tg.to_string(index=False))
print("\ntotal units passing the Methods' rule (no fake-session test): all pulses %d, isolated pulses %d; paper 39 + 32 = 71"
      % (tg.tagged_all_pulses.sum(), tg.tagged_isolated_pulses.sum()))
print("light event durations (s) and counts, first session:", dict(zip(*np.unique(np.round(E[0]["trials"]["phototagging"][1] - E[0]["trials"]["phototagging"][0], 3), return_counts=True))))
print(f"\nnotebook ran in {(time.time() - started) / 60:.1f} min")

The rule tags 1355 units with all pulses and 1444 with the isolated (1 Hz) pulses, about three
quarters of all units and twenty times the paper's 71. Without the fake-session test, which removes
units that respond to the electrical noise of the light, the rule does not pick out tagged neurons
in these units.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig3k-nt-sensor-sucrose-increase | not reproduced (right direction, p = 0.08 at best) |
| fig3l-nt-sensor-airpuff-decrease | not reproduced (right direction, p = 0.07 at best) |
| fig3-nt-sensor-early-dip | not reproduced (sucrose below zero, airpuff above) |
| fig4-recording-count | not reproduced (1902 or 1739 units against 685; 31 mice against 30) |
| fig4d-seven-clusters | blocked, methods_underspecified (7 clusters only with an 820 ms smoothing window) |
| fig5dh-behavior-clusters | blocked, metadata_insufficient (pose not on the trial clock) |
| 13 claims needing group or projection labels | blocked, metadata_insufficient |
| 20 claims on experiments not in the dandiset | blocked, data_missing |

**How the analysis differs from the paper's.** The photometry cues are taken as CS-US trials with the
US 2 s after onset, since the files mark neither the US nor unpaired cues; the photometry z-score is
run with per-trial and session baselines; the clustering smoothing window is run over a range since
MATLAB's smoothing factor cannot be reproduced; and the units are the files' units, not sorted
single units.

**What was found about the dataset.** The files carry no CRISPR or control label and no tagged
projection, which most of the paper's electrophysiology findings need; the units are not sorted
single units (frequent sub-millisecond intervals, spikes shared between units, 64 units in most
sessions); H17's spike times stop at 249 s of a 12118 s recording; the pose has no timestamps and
its nominal 15 frames per second drifts against the trial clock by tens of seconds; there are 31
mice where the paper has 30, and two of them may not be recorded on their last day; six sessions
have no neutral CS though the description says 2:1:1; the photometry series
interleaves three undescribed frame types at 60 Hz in one series, the cue events last 2 s where the
Methods give 4 s, and no event marks the US or unpaired cues. Only the NT sensor experiment of the
photometry is deposited; the behavioral cohorts, the axon-terminal and somatic photometry, the
anatomy and the slice physiology are not.